<div style="
    width:100%;
    max-width:1080px;
    margin:24px 0 32px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #c2410c 0%, #f59e0b 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h1 style="margin:0 0 14px 0; font-size:34px; line-height:1.2; font-weight:700; color:#ffffff;">El Análisis de Errores</h1>
<p style="margin:0; color:#cbd5e1; font-size:18px;">Más allá de las métricas, conviene mirar de cerca en qué casos falla el modelo y por qué. Este notebook se centra en el análisis de errores como herramienta para mejorar sistemas de NLP.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Por qué importa</div>
    <div style="color:#f8fafc;">Entender los fallos permite detectar sesgos, datos problemáticos, ambigüedades y límites reales del modelo.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Qué vas a aprender aquí</div>
    <div style="color:#f8fafc;">A interpretar errores del sistema como fuente de diagnóstico y no solo como fracasos aislados.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Regla práctica que debes recordar</div>
    <div style="color:#f8fafc;">Mejorar un modelo exige estudiar sus errores con el mismo cuidado con el que celebras sus aciertos.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#fbbf24;">Contexto mínimo</div>
    <div style="color:#f8fafc;">Analizar errores implica revisar casos concretos, patrones de fallo y posibles causas estructurales del problema.</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:36px 0 42px 0;">

In [ ]:
# Bloque 1 — Carga del modelo y del tokenizer
"""Este bloque carga un modelo preentrenado con cabeza de clasificación y su tokenizer asociado.
Este bloque importa "AutoModelForSequenceClassification" y "AutoTokenizer" de "transformers" para cargar un modelo preentrenado 
y su tokenizador correspondiente.

Aplicamos la clase "AutoModelForSequenceClassification", acompañada del método ".from_pretrained()", a la variable "modelo" para cargar un modelo preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
El parámetro "num_labels=" especifica el número de etiquetas de clasificación que el modelo debe predecir.
La variable "modelo" almacena una instancia del modelo para clasificación de secuencias. Su cabeza de clasificación se inicializa de nuevo y no se entrena en este ejemplo.

Aplicamos la clase "AutoTokenizer", acompañada del método ".from_pretrained()", a la variable "tokenizador" para cargar un tokenizador preentrenado.
El argumento "bert-base-uncased" especifica el modelo preentrenado que se desea utilizar.
La variable "tokenizador" almacena una instancia del tokenizador preentrenado lista para ser utilizada en tareas de clasificación de secuencias.
Estos son los componentes fundamentales para calcular métricas y para ejecutar análisis de errores en modelos de texto.
El modelo y el tokenizador están preentrenados para inglés, mientras que los textos del ejemplo están en español."""

from transformers import AutoModelForSequenceClassification, AutoTokenizer

modelo = AutoModelForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
tokenizador = AutoTokenizer.from_pretrained("bert-base-uncased")

modelo, tokenizador # Muestra el modelo y el tokenizador cargados para verificación.


In [ ]:
# Bloque 2 — Construcción del dataset de evaluación
"""Este bloque construye un dataset mínimo para realizar análisis de errores.
Este bloque importa "Dataset" de "datasets" para construir un dataset de evaluación mínimo.

Definimos un dataset mínimo con textos y etiquetas para la evaluación de modelos de clasificación de texto.
La variable "textos_eval" contiene una lista con los textos de entrada para la evaluación de modelos de clasificación de texto.
La variable "etiquetas_eval" contiene una lista de etiquetas correspondientes a cada uno de los textos de entrada.

Definimos el diccionario que contiene los textos y etiquetas a partir del cual se construye el dataset de evaluación.
La variable "diccionario_eval" almacena los textos y etiquetas que se utilizarán para construir el dataset de evaluación.

Definimos la lista "lista_columnas" que contiene los nombres de las columnas que se incluirán en el dataset tokenizado.
La variable "lista_columnas" contiene los nombres de las columnas que se incluirán en el dataset tokenizado.

Aplicamos la clase "Dataset", acompañada del método ".from_dict()", a la variable "dataset_eval" para crear el dataset de evaluación.
El argumento "diccionario_eval" es el diccionario definido previamente a partir del cual se construye el dataset de evaluación.
La variable "dataset_eval" contiene el dataset de evaluación sin tokenizar listo para ser tokenizado y utilizado en el análisis 
de errores de modelos de clasificación de texto. Sus cuatro ejemplos son ilustrativos y no permiten una evaluación representativa."""

from datasets import Dataset

textos_eval = ["El producto es excelente.", "La experiencia fue terrible.", "El envío llegó rápido.", "La calidad es deficiente."]
etiquetas_eval = [1, 0, 1, 0]
diccionario_eval = {"texto": textos_eval, "label": etiquetas_eval}
lista_columnas = ["input_ids", "attention_mask", "label"]

dataset_eval = Dataset.from_dict(diccionario_eval)

dataset_eval # Muestra el dataset de evaluación sin tokenizar para verificar su contenido.


In [ ]:
# Bloque 3 — Tokenización del dataset
"""Este bloque aplica tokenización al dataset de evaluación.
Definimos la función "def tokenize_batch(batch)" para tokenizar cada lote de textos del dataset de evaluación.
El parámetro "batch" es un diccionario que contiene los textos asociados a la clave "texto" del diccionario definido previamente.

Esta función retorna la llamada al tokenizador "tokenizador()" con los textos del lote, aplicando padding y truncado.
El argumento "batch["texto"]" contiene los textos del lote asociados a la clave "texto".
El parámetro "padding=" rellena las secuencias hasta la longitud de la más larga del lote.
El parámetro "truncation=" trunca las secuencias que exceden la longitud máxima del modelo.

Aplicamos el método ".map()" al dataset de evaluación para tokenizar sus textos.
El argumento "tokenize_batch" es la función que se aplica a cada lote para realizar la tokenización.
El parámetro "batched=" indica que la función se aplicará a lotes de ejemplos en lugar de a ejemplos individuales.
La variable "dataset_eval_tokenizado" almacena el dataset de evaluación con los campos necesarios para el modelo.

Aplicamos el método ".set_format()" al dataset de evaluación tokenizado para establecer el formato de salida.
El parámetro "type=" indica con el valor "torch" que los datos se devolverán como tensores de PyTorch.
El parámetro "columns=" especifica las columnas que se incluirán en la salida, en este caso, las columnas definidas en la lista "lista_columnas".

La variable "dataset_eval_tokenizado" almacena el dataset con los campos necesarios para el modelo.
Esta operación permite que el dataset tokenizado se pueda utilizar directamente en modelos de PyTorch para evaluación, 
entrenamiento o inferencia."""

def tokenize_batch(batch):
    return tokenizador(batch["texto"], padding=True, truncation=True)

dataset_eval_tokenizado = dataset_eval.map(tokenize_batch, batched=True)
dataset_eval_tokenizado.set_format(type="torch", columns=lista_columnas)

dataset_eval_tokenizado # Muestra el dataset tokenizado con los campos necesarios para el modelo.


In [ ]:
# Bloque 4 — Obtención de predicciones del modelo
"""Este bloque obtiene las predicciones del modelo sobre el dataset tokenizado.
Este bloque importa "torch" para trabajar con tensores en PyTorch.

Asignamos el atributo ".logits" de la salida de la llamada al modelo a la variable "logits" para 
obtener las puntuaciones del modelo.

El parámetro "input_ids=" contiene los identificadores de los tokens del texto de entrada.
El valor "dataset_eval_tokenizado[:]["input_ids"]" contiene los identificadores de los tokens del texto de entrada.

El parámetro "attention_mask=" indica qué tokens deben ser atendidos por el modelo y cuáles deben ser ignorados.
El valor "dataset_eval_tokenizado[:]["attention_mask"]" contiene las máscaras de atención correspondientes al texto de entrada.
La variable "logits" almacena las puntuaciones del modelo antes de seleccionar la clase con mayor valor.

Aplicamos la función "torch.argmax()" a los logits y asignamos las etiquetas predichas a la variable "predicciones".
El argumento "logits" corresponde a las puntuaciones de salida del modelo; argmax selecciona un índice y no es una función de activación.
El parámetro "dim=" indica con el valor "-1" que se debe aplicar argmax a lo largo de la última dimensión de los logits.
La variable "predicciones" contiene en forma de tensor las etiquetas predichas por el modelo.
Esta llamada no desactiva explícitamente el cálculo de gradientes."""   

import torch

logits = modelo(
    input_ids=dataset_eval_tokenizado[:]["input_ids"],  # El operador "[:][x]" selecciona todos los ejemplos de la columna seleccionada del dataset tokenizado.         
    attention_mask=dataset_eval_tokenizado[:]["attention_mask"]  
).logits

predicciones = torch.argmax(logits, dim=-1)

predicciones # Muestra las etiquetas predichas por el modelo para el dataset de evaluación.


In [ ]:
# Bloque 5 — Identificación de errores
"""Este bloque identifica los ejemplos donde el modelo falla.
Utilizamos una comprensión de lista "[]" para obtener los índices de los ejemplos mal clasificados.

La subexpresión "i for i" indica que se seleccionan los índices de los ejemplos que cumplen la condición especificada 
después de la palabra clave if.

La subexpresión "i, (p, y) in enumerate(x)" indica que se itera sobre los elementos de "x" junto con sus índices, 
donde "x" es el iterador de pares (p, y) obtenido mediante "zip()" aplicado a "predicciones" y "dataset_eval_tokenizado["label"]".

La subexpresión "zip(predicciones, dataset_eval_tokenizado["label"])" crea un iterador de pares (p, y), donde "p" es la 
predicción del modelo e "y" es la etiqueta real correspondiente.

La subexpresión "if p != y" indica que se seleccionan los índices de los ejemplos donde la predicción "p" es diferente    
de la etiqueta real "y".

Utilizamos una segunda comprensión de lista "[]" para obtener los detalles de los ejemplos mal clasificados a partir de 
los índices almacenados en la variable "indices_errores".

La subexpresión "{"texto": textos_eval[i], "etiqueta_real": etiquetas_eval[i], "prediccion": int(predicciones[i]), 
"logits": logits[i].tolist()}" crea un diccionario con los detalles del error correspondiente al índice i.

La subexpresión "for i in indices_errores" indica que se itera sobre los índices de los ejemplos mal clasificados de la variable 
"indices_errores" para obtener los detalles de los ejemplos mal clasificados.

La variable "indices_errores" contiene las posiciones de los ejemplos mal clasificados obtenidas mediante la comprensión de lista.
La variable "errores" almacena los textos, etiquetas reales, predicciones incorrectas y logits correspondientes.
Esta operación permite identificar y analizar los errores cometidos por el modelo en los ejemplos de evaluación."""

indices_errores = [
    i for i, (p, y) in enumerate(zip(predicciones, dataset_eval_tokenizado["label"]))
    if p != y
]

errores = [
    {
        "texto": textos_eval[i],             # Obtiene el valor asociado a la clave "texto" del ejemplo.
        "etiqueta_real": etiquetas_eval[i],  # Obtiene el valor asociado a la clave "etiqueta_real" del ejemplo.
        "prediccion": int(predicciones[i]),  # Obtiene el valor asociado a la clave "prediccion" del ejemplo.
        "logits": logits[i].tolist()         # Obtiene el valor asociado a la clave "logits" del ejemplo como una lista.
    }
    for i in indices_errores
]

indices_errores, errores # Muestra los índices de los ejemplos mal clasificados y los detalles de los errores.


In [ ]:
# Bloque 6 — Organización del análisis
"""Este bloque organiza los resultados del análisis de errores.
La variable "resultado" almacena en forma de diccionario la lista de ejemplos mal clasificados junto con información 
relevante para su estudio."""

resultado = {
    "errores_detectados": errores
}

resultado # Muestra el diccionario con los errores detectados.


<div style="
    width:100%;
    max-width:1080px;
    margin:40px 0 24px 0;
    background:linear-gradient(180deg, #1f2937 0%, #0f172a 100%);
    border:1px solid #334155;
    border-radius:22px;
    box-shadow:0 18px 38px rgba(15,23,42,0.24);
    overflow:hidden;
    color:#e6edf7;
">

<div style="
    height:8px;
    background:linear-gradient(90deg, #0f766e 0%, #2dd4bf 100%);
"></div>

<div style="
    max-width:860px;
    padding:34px 38px 36px 38px;
    margin-left:0;
    margin-right:auto;
    font-family:'Aptos', 'Segoe UI Variable', 'Segoe UI', sans-serif;
    font-size:19px;
    line-height:1.78;
    color:#e6edf7;
">

<div style="margin-bottom:22px;">
<h2 style="margin:0 0 14px 0; font-size:30px; line-height:1.2; font-weight:700; color:#ffffff;">Conclusiones</h2>
<p style="margin:0; color:#cbd5e1; font-size:18px;">En este notebook has visto que analizar fallos concretos ofrece información que las métricas agregadas no siempre muestran.</p>
</div>

<div style="display:grid; gap:14px;">
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Qué conviene recordar</div>
    <div style="color:#f8fafc;">Los errores bien estudiados orientan mejor las mejoras del sistema que una simple intuición general.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Error frecuente a evitar</div>
    <div style="color:#f8fafc;">Quedarse solo con el resultado numérico y no mirar ejemplos reales donde el modelo se equivoca.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente paso natural</div>
    <div style="color:#f8fafc;">Detenerse en la Inferencia Optimizada para dar continuidad directa al hilo del temario.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Conexión con el siguiente concepto</div>
    <div style="color:#f8fafc;">El siguiente notebook se centra en la Inferencia Optimizada y amplía de forma natural lo que acabas de trabajar.</div>
  </div>
  <div style="padding:16px 18px; border:1px solid #475569; border-radius:14px; background:rgba(148,163,184,0.10);">
    <div style="margin:0 0 6px 0; font-size:14px; font-weight:700; letter-spacing:0.08em; text-transform:uppercase; color:#5eead4;">Siguiente notebook del temario</div>
    <div style="color:#f8fafc;">08_Inferencia_optimizada/01_La_inferencia_optimizada.ipynb</div>
  </div>
</div>

</div>

</div>

<hr style="border:0; border-top:1px solid #64748b; opacity:0.22; margin:42px 0 24px 0;">